# GIS Transportation & Vehicle Routing Optimization
### Executable Specification & Prototype Notebook

---

**Table of Contents**

| # | Section | Description |
|---|---------|-------------|
| 1 | [Executive Summary](#1-executive-summary) | Problem statement and project goals |
| 2 | [Data Sources](#2-data-sources) | Road network, APIs, and databases used |
| 3 | [Core Formulas & Methodology](#3-core-formulas--methodology) | All spatial/math formulas in the system |
| 4 | [Prototype Code](#4-prototype-code) | Self-contained proof-of-concept cells |
| 5 | [System Specification & Tech Handoff](#5-system-specification--tech-handoff) | Inputs, outputs, constraints, endpoints |

---

## 1. Executive Summary

### Problem

Fleet operators managing 50–100+ deliveries across Maharashtra:
- **Assign parcels** to 10 heterogeneous vehicles
- **Minimize cost** (distance × per-km rate)
- **Respect constraints** — vehicle capacity (kg), driver shift windows, delivery deadlines
- **Avoid hazards** — monsoon flooding, traffic congestion
- **Use real roads** — not straight-line ("as the crow flies") distances

Manual planning leads to sub-optimal routes, missed deadlines, wasted fuel, and dangerous roads.

### Solution

This system solves the **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)**
using real road data. Here's what it does:

| Feature | How It Works |
|---------|-------------|
| **Real Road Distances** | pgRouting on 843K road segments (no straight lines) |
| **Live Traffic** | Google Routes API or TomTom → congestion factor per road segment |
| **Monsoon Weather** | OpenWeatherMap → 3×–10× penalty on flooded roads |
| **Dual Optimizer** | Google Fleet Routing (primary) + OR-Tools (fallback) |
| **Auto Re-optimize** | Every 10 min: refresh traffic, re-order stops |
| **Geocoding** | Ola Krutrim Maps API → Nominatim/OSM fallback (with JSON cache) |
| **Visualization** | Interactive map with color-coded routes (🟢🟡🟠🔴 by congestion) |

**Input:** CSV/Excel parcels delivery sheet → 
**Output:** Optimized routes with real road geometries, costs, and schedules.

## 2. Data Sources

### 2.1 Road Network (PostGIS + pgRouting)

| Property | Value |
|----------|-------|
| **Source** | OpenStreetMap (Maharashtra extract) |
| **Table** | `vector.road_maharashtra` (~843K road segments) |
| **Key Columns** | `gid`, `source`, `target`, `cost_s` (seconds), `reverse_cost_s`, `geom` (LineString SRID 4326) |
| **Traffic Columns** | `traffic_factor`, `live_cost_s`, `live_reverse_cost_s`, `last_traffic_update` |
| **Extensions** | PostGIS 3.x, pgRouting 3.x |
| **Indexes** | GiST on `geom`, B-tree on `source`/`target` |

### 2.2 Pre-computed Node Table

| Property | Value |
|----------|-------|
| **Table** | `vector.main_road_nodes` |
| **Purpose** | All nodes in the largest connected component (component 11) |
| **Benefit** | Eliminates repeated `pgr_connectedComponents` calls during snapping |
| **Index** | GiST on `geom` for fast KNN queries |

### 2.3 Traffic APIs

| API | Endpoint | Auth | What It Returns |
|-----|----------|------|-----------------|
| Google Routes | `routes.googleapis.com/directions/v2:computeRoutes` | OAuth2 Service Account | `duration` vs `staticDuration` |
| TomTom Flow v4 | `api.tomtom.com/traffic/services/4/flowSegmentData/...` | API Key | `currentSpeed` / `freeFlowSpeed` |

Toggle via `TRAFFIC_SOURCE=google|tomtom` in `.env`.

### 2.4 Weather API

| Property | Value |
|----------|-------|
| **API** | OpenWeatherMap Current Weather (`api.openweathermap.org/data/2.5/weather`) |
| **Thresholds** | IMD classification: Light <2.5 mm/hr, Moderate 2.5–7.5, Heavy ≥7.5 |
| **Simulation** | `WEATHER_SIMULATE_RAIN=true` to simulate to rain when it is not actually raining |

### 2.5 Geocoding APIs

| Priority | API | Details |
|----------|-----|--------|
| Primary | Ola Maps | `api.olamaps.io/places/v1/geocode` — parallel batch (5 concurrent) |
| Fallback | Nominatim/OSM | `nominatim.openstreetmap.org/search` — rate-limited 1 req/sec |
| Cache | Local JSON | `geocode_cache.json` — keyed by raw address string |

### 2.6 Optimization APIs

| Engine | Details |
|--------|---------|
| Google Fleet Routing | `routeoptimization.googleapis.com/v1/projects/{id}:optimizeTours` — OAuth2, delivery-only mode |
| OR-Tools (Local) | `ortools==9.11.4210` — SAVINGS heuristic + Guided Local Search |

## 3. Core Formulas & Methodology

This section contains **every formula used in the system**, organized by pipeline stage.

---

### 3.1 Station Snapping (Address → Road Node)

Every delivery address must be "snapped" to the nearest node on the road graph.

**Formula (PostGIS KNN):**

$$node_i = \arg\min_{n \in \text{main\_road\_nodes}} \; d_{KNN}(n.\text{geom}, \; \text{ST\_Point}(lon_i, lat_i))$$

- Uses PostGIS KNN operator `<->` with GiST index — **O(log N)** per lookup
- Batch insert all stations via temp table + single spatial JOIN (not N individual queries)

---

### 3.2 Distance Matrix (pgRouting Dijkstra)

The VRP solver needs an **N × N matrix** where N = 1 warehouse + S stations.

**Formula:**

$$D(i, j) = \text{pgr\_dijkstraCost}(i, j) \quad \text{[seconds, using } \texttt{COALESCE(live\_cost\_s, cost\_s)}\text{]}$$

- **Spatial filter:** Only load road segments within `ST_Expand(bbox, 0.3°)` (~33 km buffer)
- Results stored in `vector.distance_matrix (start_vid, end_vid, agg_cost)`
- `live_cost_s = cost_s × traffic_factor` — traffic penalties propagate into Dijkstra edge weights

---

### 3.3 Traffic Factor Calculation

Traffic congestion is a dimensionless multiplier applied to road travel times.

| Source | Formula |
|--------|---------|
| **Google Routes API** | $\text{traffic\_factor} = \frac{\text{duration (with traffic)}}{\text{staticDuration (free-flow)}}$ |
| **TomTom Flow API** | $\text{traffic\_factor} = \frac{\text{freeFlowSpeed}}{\text{currentSpeed}}$ |

**Bounds:** $\text{traffic\_factor} \in [0.8, 10.0]$

| Factor Range | Congestion Level | Map Color |
|---|---|---|
| ≤ 1.1 | Free flow | 🟢 Green (`#22C55E`) |
| 1.1 – 1.5 | Light | 🟡 Yellow (`#EAB308`) |
| 1.5 – 2.0 | Moderate | 🟠 Orange (`#F97316`) |
| > 2.0 | Heavy | 🔴 Red (`#DC2626`) |


---

### 3.4 Weather Penalty (IMD Classification)

Rainfall intensity is classified per India Meteorological Department (IMD) standards:

| Severity | Rain (mm/hr) | Road Penalty Factor | Effect |
|---|---|---|---|
| None | < 2.5 | 1.0× | No change |
| Moderate | 2.5 – 7.5 | 3.0× | pgRouting prefers alternate roads |
| Heavy | ≥ 7.5 | 10.0× | pgRouting strongly avoids these roads |

**Combined Road Cost:**
```
live_cost_s = cost_s × max(traffic_factor, weather_penalty_factor)
```

**Simulation Mode Seed (deterministic):**

$$\text{seed} = \lfloor |lat \times 10000| \times |lon \times 100| \rfloor \mod 100$$

- seed < 15 → Heavy rain (~15%)
- 15 ≤ seed < 40 → Moderate rain (~25%)
- seed ≥ 40 → Clear (~60%)

---

### 3.5 VRP Formulation (CVRPTW)

**Objective — Minimize total cost:**

$$\min \sum_{v=1}^{V} \sum_{(i,j) \in \text{route}_v} c_v \cdot D(i,j)$$

where $c_v$ = cost per km for vehicle $v$, $D(i,j)$ = road distance in km.

**Constraints:**

| # | Constraint | Formula | Penalty |
|---|-----------|---------|--------|
| 1 | **Capacity** | $\sum_{i \in \text{route}_v} w_i \leq C_v$ | 1,000,000 per kg over |
| 2 | **Time Windows** | $a_i \leq t_i \leq b_i$ | 100,000 per minute late |
| 3 | **Shift Limits** | $S_v^{start} \leq t_v \leq S_v^{end}$ | 50,000 per minute overtime |
| 4 | **Cross-midnight** | If $S_v^{end} < S_v^{start}$, add 1440 | — |
| 5 | **Service time** | Each stop adds $s_i$ min (default 10) | — |
| 6 | **Warehouse loading** | 10 min fixed buffer at depot | — |
| 7 | **Drop penalty** | 1,000,000,000 per undelivered parcel | Effectively infinite |

**Travel time conversion (seconds → minutes):**

$$t_{ij} = \max\left(1, \; \text{round}\left(\frac{D_{ij}^{seconds}}{60}\right)\right) + s_i$$

**OR-Tools Configuration:**

| Parameter | Value |
|-----------|-------|
| First Solution | `SAVINGS` heuristic |
| Metaheuristic | `GUIDED_LOCAL_SEARCH` |
| Time limit | 10 seconds |
| Slack (waiting) | up to 120 minutes |

---

### 3.6 Google Route Optimization (Delivery-Only Mode)

When `USE_GOOGLE_OPTIMIZATION=true`, the system uses Google's Fleet Routing API.

**Critical Design Decision — Delivery-Only Mode:**
- Shipments use `deliveries` with `loadDemands.weight`
- Total assigned weight per vehicle never exceeds `loadLimits.weight.maxLoad`

---

### 3.7 Route Geometry Generation (pgRouting → GeoJSON)

After VRP solution, actual road geometries are generated for map display:

1. For each vehicle's stop sequence: $[depot, s_1, s_2, \ldots, s_k, depot]$
2. For each consecutive pair $(s_i, s_{i+1})$, run `pgr_dijkstra` with spatial filter
3. JOIN result edges with `vector.road_maharashtra` for LineString geometries
4. `ST_Collect` + `ST_Multi` to merge into per-segment MultiLineString
5. Record `avg_traffic_factor` per segment for color-coded visualization

**Batch optimization:** Global bounding box computed once; all vehicles processed
with `LATERAL JOIN` — 8 queries instead of 56.

---

### 3.8 Auto Re-optimization (TSP per Vehicle)

Every 10 minutes, the system re-orders stops within each vehicle:

1. Read current vehicle→parcel assignments from DB (fixed)
2. Refresh traffic data (new API calls)
3. For each vehicle with ≥2 stops, solve a mini-TSP using OR-Tools or Google
4. Compare new order vs old order → flag rerouted vehicles
5. Regenerate road geometries with fresh traffic colors

**Key constraint:** Parcel-to-vehicle assignments are **NEVER** changed.
Only the stop ordering within each vehicle is re-optimized.

---

### 3.9 Road Distance Calculation (Real km)

Route distances are calculated from saved geometries using PostGIS geography:

$$\text{distance\_km}_v = \frac{\sum_{s \in \text{segments}_v} \text{ST\_Length}(s.\text{geom}::geography)}{1000}$$

**Operational cost:**

$$\text{cost}_v = \text{distance\_km}_v \times c_v$$

---

### 3.10 Delivery Status Classification

| Status | Condition |
|--------|-----------|
| `IN_BUFFER` | Arrival ≤ deadline − 60 min |
| `ON TIME` | deadline − 60 < arrival ≤ deadline |
| `LATE` | Arrival > deadline |

---

### 3.11 Geocoding Pipeline

```
Input Address → Cache Check → Ola Maps API → Nominatim Fallback → (lat, lon)
```

- **Ola Maps:** Parallel batch (5 concurrent)
- **Nominatim:** Sequential (1 req/sec rate limit)
- **Cache:** JSON file, keyed by raw address string
- **Validation:** India bounds (6.5°–35.5°N, 68°–97.5°E)

## 4. Prototype Code

Each cell below is a **self-contained proof-of-concept** that validates
one piece of the methodology from Section 3.

> **Note:** These are demonstration prototypes, not production code.
> The actual production code lives in the `backend/` directory.

### 4.1 — Station Snapping: Address → Nearest Road Node

**What this does:** Snaps a delivery coordinate to the nearest node on the
road network using PostGIS KNN operator (`<->`).

**Related formula:** Section 3.1

In [1]:
import psycopg2
from psycopg2.extras import RealDictCursor

# --- Configuration (replace with your actual DB credentials) ---
DB_CONFIG = {
    "dbname": "your_db",
    "user": "your_user",
    "password": "your_password",
    "host": "localhost",
    "port": "5432"
}

def snap_to_road_node(lat: float, lon: float, conn) -> dict:
    """
    Snap a lat/lon to the nearest road network node using KNN.
    
    Uses the PostGIS <-> operator on the GiST-indexed main_road_nodes table.
    Returns the nearest node_id, its coordinates, and snap distance in meters.
    """
    cur = conn.cursor(cursor_factory=RealDictCursor)
    cur.execute("""
        SELECT node_id,
               ST_X(geom) AS node_lon,
               ST_Y(geom) AS node_lat,
               ST_Distance(
                   geom::geography, 
                   ST_SetSRID(ST_Point(%s, %s), 4326)::geography
               ) AS dist_meters
        FROM vector.main_road_nodes
        ORDER BY geom <-> ST_SetSRID(ST_Point(%s, %s), 4326)
        LIMIT 1;
    """, (lon, lat, lon, lat))
    result = dict(cur.fetchone())
    cur.close()
    return result

# --- Example (uncomment to run with a real database) ---
# conn = psycopg2.connect(**DB_CONFIG)
# sample = {"lat": 19.0760, "lon": 72.8777}  # Mumbai CST
# snapped = snap_to_road_node(sample["lat"], sample["lon"], conn)
# print(f"Input:    ({sample['lat']}, {sample['lon']})")
# print(f"Snapped:  ({snapped['node_lat']:.6f}, {snapped['node_lon']:.6f})")
# print(f"Distance: {snapped['dist_meters']:.1f} m")
# conn.close()

### 4.2 — Distance Matrix via pgRouting (Spatial-Filtered Dijkstra)

**What this does:** Calculates all-pairs shortest-path costs (in seconds)
using Dijkstra's algorithm on the real road network, with a spatial
bounding box filter to avoid loading all 843K roads.

**Related formula:** Section 3.2

In [4]:
def calculate_pairwise_distance(conn, node_ids: list) -> dict:
    """
    Calculate all-pairs shortest path cost using pgr_dijkstraCost with spatial filter.
    
    Steps:
    1. Get bounding box of all input nodes
    2. Expand bbox by 0.3° (~33km) to include surrounding roads
    3. Run Dijkstra only on roads within this bbox
    4. Return dict of {(start_node, end_node): cost_seconds}
    """
    cur = conn.cursor()

    # Get bounding box of nodes
    cur.execute("""
        SELECT ST_Extent(geom)
        FROM vector.main_road_nodes
        WHERE node_id = ANY(%s)
    """, (node_ids,))
    extent = cur.fetchone()[0]

    # Run Dijkstra with spatial filter (0.3 degree buffer ~33km)
    cur.execute("""
        SELECT start_vid, end_vid, agg_cost
        FROM pgr_dijkstraCost(
            format('SELECT gid AS id, source, target, 
                           COALESCE(live_cost_s, cost_s) AS cost
                    FROM vector.road_maharashtra
                    WHERE geom && ST_Expand(
                        ST_SetSRID(%%L::box2d::geometry, 4326), 0.3)', %s),
            %s::bigint[],
            %s::bigint[],
            directed := false
        )
    """, (extent, node_ids, node_ids))

    dist_dict = {}
    for row in cur.fetchall():
        dist_dict[(int(row[0]), int(row[1]))] = float(row[2])

    cur.close()
    return dist_dict

# --- Example (uncomment to run) ---
# conn = psycopg2.connect(**DB_CONFIG)
# sample_nodes = [123456, 234567, 345678]  # Replace with real node IDs
# distances = calculate_pairwise_distance(conn, sample_nodes)
# for (a, b), cost_s in sorted(distances.items()):
#     print(f"  Node {a} → {b}: {cost_s:.0f}s ({cost_s/60:.1f} min)")
# conn.close()

### 4.3 — Traffic Factor Calculation (Google Routes API)

**What this does:** Queries Google Routes API to compare live travel time
vs static (free-flow) travel time for a short probe route near a station.

**Related formula:** Section 3.3 — `traffic_factor = duration / staticDuration`

In [1]:
import httpx
import asyncio

async def get_traffic_factor_google(lat: float, lon: float, access_token: str) -> float:
    """
    Estimate congestion factor using Google Routes API.
    
    Strategy:
    - Create a short probe trip (~500m north of the station)
    - Compare duration (with live traffic) vs staticDuration (free-flow)
    - Factor = duration / staticDuration
    - Clamped to [0.8, 10.0]
    """
    ROUTES_URL = "https://routes.googleapis.com/directions/v2:computeRoutes"

    body = {
        "origin": {"location": {"latLng": {"latitude": lat, "longitude": lon}}},
        "destination": {"location": {"latLng": {"latitude": lat + 0.005, "longitude": lon}}},
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE"
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {access_token}",
        "X-Goog-FieldMask": "routes.duration,routes.staticDuration"
    }

    async with httpx.AsyncClient() as client:
        response = await client.post(ROUTES_URL, json=body, headers=headers, timeout=8)

    if response.status_code == 200:
        routes = response.json().get("routes", [])
        if routes:
            duration = float(routes[0].get("duration", "0s").rstrip("s"))
            static = float(routes[0].get("staticDuration", "0s").rstrip("s"))
            if static > 0:
                factor = round(min(max(duration / static, 0.8), 10.0), 3)
                return factor
    return 1.0  # Default: free flow

# --- Example (uncomment to run) ---
# factor = asyncio.run(get_traffic_factor_google(19.076, 72.877, "your_oauth2_token"))
# print(f"Traffic factor at Mumbai CST: {factor}x")

### 4.4 — Weather Penalty Classification (IMD Thresholds)

**What this does:** Classifies rainfall intensity using India Meteorological
Department standards and returns the appropriate road penalty factor.

**Related formula:** Section 3.4

In [ ]:
def classify_weather_penalty(rain_mm_per_hr: float) -> dict:
    """
    Classify rainfall and return penalty factor per IMD thresholds.
    
    Thresholds (India Meteorological Department):
    - Light:    < 2.5 mm/hr  → 1.0x (no penalty)
    - Moderate: 2.5–7.5 mm/hr → 3.0x (roads slow)
    - Heavy:    ≥ 7.5 mm/hr  → 10.0x (waterlogging risk)
    """
    RAIN_LIGHT = 2.5
    RAIN_MODERATE = 7.5

    if rain_mm_per_hr >= RAIN_MODERATE:
        return {"severity": "heavy", "penalty_factor": 10.0,
                "description": f"Heavy Rain ({rain_mm_per_hr:.1f} mm/hr)"}
    elif rain_mm_per_hr >= RAIN_LIGHT:
        return {"severity": "moderate", "penalty_factor": 3.0,
                "description": f"Moderate Rain ({rain_mm_per_hr:.1f} mm/hr)"}
    else:
        return {"severity": "none", "penalty_factor": 1.0,
                "description": "Clear / Light"}

# --- Test all severity levels ---
test_cases = [0.5, 3.2, 8.5, 15.0]
for rain in test_cases:
    result = classify_weather_penalty(rain)
    print(f"  Rain={rain:5.1f} mm/hr → {result['severity']:>8s} | "
          f"Penalty={result['penalty_factor']:4.1f}x | {result['description']}")

  Rain=  0.5 mm/hr →     none | Penalty= 1.0x | Clear / Light
  Rain=  3.2 mm/hr → moderate | Penalty= 3.0x | Moderate Rain (3.2 mm/hr)
  Rain=  8.5 mm/hr →    heavy | Penalty=10.0x | Heavy Rain (8.5 mm/hr)
  Rain= 15.0 mm/hr →    heavy | Penalty=10.0x | Heavy Rain (15.0 mm/hr)


### 4.5 — Monsoon Simulation (Deterministic Seeding)

**What this does:** Simulates monsoon weather for testing by using a
coordinate-based hash so the **same station always gets the same weather**.

**Related formula:** Section 3.4 — seed formula

**Bug fix note:** The original implementation used `random.uniform()` without seeding
the RNG per coordinate, so rain amounts were non-deterministic. This version
seeds `random` with the coordinate hash for full determinism.

In [1]:
import random

def simulate_monsoon(lat: float, lon: float) -> dict:
    """
    Deterministic monsoon simulation using coordinate-seeded hash.
    
    Same (lat, lon) always produces the same weather result.
    Distribution: ~15% heavy, ~25% moderate, ~60% clear.
    
    Seed formula: seed = floor(|lat * 10000| * |lon * 100|) mod 100
    """
    seed = int(abs(lat * 10000) * abs(lon * 100)) % 100
    
    # Seed the RNG so rain amounts are also deterministic per coordinate
    rng = random.Random(seed)

    if seed < 15:        # ~15% heavy rain
        rain_mm = round(rng.uniform(8.0, 18.0), 1)
        return {"severity": "heavy", "penalty_factor": 10.0,
                "rain_mm": rain_mm, "seed": seed}
    elif seed < 40:      # ~25% moderate rain
        rain_mm = round(rng.uniform(3.0, 7.0), 1)
        return {"severity": "moderate", "penalty_factor": 3.0,
                "rain_mm": rain_mm, "seed": seed}
    else:                # ~60% clear
        return {"severity": "none", "penalty_factor": 1.0,
                "rain_mm": 0.0, "seed": seed}

# --- Validate distribution over sample coordinates ---
heavy = moderate = clear = 0
for i in range(100):
    lat = 18.5 + (i * 0.01)
    lon = 72.5 + (i * 0.005)
    r = simulate_monsoon(lat, lon)
    if r["severity"] == "heavy": heavy += 1
    elif r["severity"] == "moderate": moderate += 1
    else: clear += 1

print("=== Monsoon Simulation Results ===")
print(f"Distribution over 100 points:")
print(f"  Heavy:    {heavy}%")
print(f"  Moderate: {moderate}%")
print(f"  Clear:    {clear}%")

# Show a few samples
print("\nSample stations:")
for i in [0, 5, 10]:
    lat = 18.5 + (i * 0.01)
    lon = 72.5 + (i * 0.005)
    r = simulate_monsoon(lat, lon)
    print(f"  ({lat:.2f}, {lon:.3f}) → seed={r['seed']:2d} → "
          f"{r['severity']:>8s} | Rain={r['rain_mm']:4.1f} mm/hr | Penalty={r['penalty_factor']:.1f}x")

=== Monsoon Simulation Results ===
Distribution over 100 points:
  Heavy:    42%
  Moderate: 0%
  Clear:    58%

Sample stations:
  (18.50, 72.500) → seed= 0 →    heavy | Rain=16.4 mm/hr | Penalty=10.0x
  (18.55, 72.525) → seed=50 →     none | Rain= 0.0 mm/hr | Penalty=1.0x
  (18.60, 72.550) → seed= 0 →    heavy | Rain=16.4 mm/hr | Penalty=10.0x


### 4.6 — VRP Solver (OR-Tools Proof-of-Concept)

**What this does:** Demonstrates the core OR-Tools CVRPTW formulation
with capacity constraints, time windows, and shift limits using a
small synthetic dataset (5 deliveries, 2 vehicles).

**Related formulas:** Section 3.5

In [2]:
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

def solve_mini_vrp():
    """
    Minimal VRP with 5 deliveries, 2 vehicles, capacity + time windows.
    
    This mirrors the production solver logic:
    - SAVINGS heuristic for initial solution
    - GUIDED_LOCAL_SEARCH metaheuristic
    - Soft time windows (penalty for late delivery)
    - Vehicle shift constraints
    - Drop penalty (effectively infinite)
    """
    # Synthetic distance matrix (in seconds)
    #              Depot  S1    S2    S3    S4    S5
    dist_matrix = [
        [0,   600,  900,  1200, 800,  1500],  # Depot
        [600,  0,    400,  700,  500,  1000],  # Station 1
        [900,  400,  0,    300,  600,  800],   # Station 2
        [1200, 700,  300,  0,    900,  500],   # Station 3
        [800,  500,  600,  900,  0,    700],   # Station 4
        [1500, 1000, 800,  500,  700,  0],     # Station 5
    ]
    demands       = [0, 20, 15, 25, 10, 30]       # kg per station
    service_times = [10, 10, 10, 10, 10, 10]       # minutes per stop
    time_windows  = [(0, 1440), (420, 720), (420, 600), (480, 900), (420, 780), (540, 1020)]
    capacities    = [60, 50]                        # kg per vehicle
    shifts        = [(420, 1080), (480, 1020)]      # minutes from midnight

    size = len(dist_matrix)
    num_vehicles = 2

    manager = pywrapcp.RoutingIndexManager(size, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    # Time callback: travel time (seconds→minutes) + service time
    def time_cb(from_idx, to_idx):
        f, t = manager.IndexToNode(from_idx), manager.IndexToNode(to_idx)
        travel_min = max(1, round(dist_matrix[f][t] / 60)) if f != t else 0
        return travel_min + service_times[f]

    transit_cb = routing.RegisterTransitCallback(time_cb)
    routing.AddDimension(transit_cb, 120, 1500, False, 'Time')  # 120 min slack
    time_dim = routing.GetDimensionOrDie('Time')

    # Soft time windows (penalize late delivery, don't prevent it)
    for i in range(1, size):
        idx = manager.NodeToIndex(i)
        time_dim.CumulVar(idx).SetRange(0, 1500)
        time_dim.SetCumulVarSoftUpperBound(idx, time_windows[i][1], 100000)

    # Vehicle shift constraints
    for v in range(num_vehicles):
        s, e = shifts[v]
        time_dim.CumulVar(routing.Start(v)).SetRange(s, e)
        time_dim.CumulVar(routing.End(v)).SetRange(s, e + 60)  # 60 min overtime

    # Capacity constraint
    def demand_cb(idx):
        return demands[manager.IndexToNode(idx)]
    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(demand_cb), 0, capacities, True, 'Cap')

    # Drop penalty (effectively infinite — prefer delivering everything)
    for i in range(1, size):
        routing.AddDisjunction([manager.NodeToIndex(i)], 1_000_000_000)

    # Arc cost (minimize distance)
    cost_cb = routing.RegisterTransitCallback(
        lambda f, t: dist_matrix[manager.IndexToNode(f)][manager.IndexToNode(t)])
    routing.SetArcCostEvaluatorOfAllVehicles(cost_cb)

    # Solve
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.SAVINGS
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.seconds = 5

    solution = routing.SolveWithParameters(params)
    if not solution:
        print("No solution found!")
        return

    # Print results
    print("=== Mini VRP Solution ===")
    for v in range(num_vehicles):
        idx = routing.Start(v)
        route = []
        while not routing.IsEnd(idx):
            node = manager.IndexToNode(idx)
            t = solution.Min(time_dim.CumulVar(idx))
            route.append(f"S{node}@{t//60:02d}:{t%60:02d}")
            idx = solution.Value(routing.NextVar(idx))
        t_end = solution.Min(time_dim.CumulVar(idx))
        route.append(f"Depot@{t_end//60:02d}:{t_end%60:02d}")
        load = solution.Min(routing.GetDimensionOrDie('Cap').CumulVar(idx))
        print(f"  Vehicle {v+1} [{capacities[v]}kg]: {' → '.join(route)} | Load={load}kg")

solve_mini_vrp()

=== Mini VRP Solution ===
  Vehicle 1 [60kg]: S0@07:00 → S1@07:20 → S2@07:37 → S3@07:52 → Depot@08:22 | Load=60kg
  Vehicle 2 [50kg]: S0@08:00 → S5@08:35 → S4@08:57 → Depot@09:20 | Load=40kg


### 4.7 — Batch Traffic Update (Spatial Join)

**What this does:** Demonstrates the spatial batch join that applies
traffic factors to road segments near multiple stations in **ONE SQL query**
instead of N individual updates.

**Related formula:** Section 3.3 — spatial application

In [ ]:
def batch_traffic_update_sql(updates: list) -> str:
    """
    Generate SQL for batch updating traffic factors on road segments.
    
    This is the key performance optimization: instead of running N individual
    UPDATE queries (one per station), we:
    1. Create a temp table with all update points
    2. Add a spatial index on it
    3. Run ONE spatial join to update all affected road segments
    
    Args:
        updates: List of (lat, lon, factor, radius_km) tuples
    Returns:
        SQL string (for review, not direct execution)
    """
    sql = """
    -- 1. Create temp table with update points
    CREATE TEMP TABLE _tmp_traffic_updates (
        lat DOUBLE PRECISION, lon DOUBLE PRECISION,
        factor REAL, radius_deg DOUBLE PRECISION
    );

    -- 2. Insert all points (radius_km * 0.01 ≈ degrees at ~19°N)
    INSERT INTO _tmp_traffic_updates VALUES
    """
    values = []
    for lat, lon, factor, radius_km in updates:
        values.append(f"    ({lat}, {lon}, {factor}, {radius_km * 0.01})")
    sql += ",\n".join(values) + ";\n\n"

    sql += """
    -- 3. Add geometry column + spatial index
    ALTER TABLE _tmp_traffic_updates ADD COLUMN geom geometry(Point, 4326);
    UPDATE _tmp_traffic_updates SET geom = ST_SetSRID(ST_Point(lon, lat), 4326);
    CREATE INDEX ON _tmp_traffic_updates USING GIST (geom);

    -- 4. Single spatial join: update roads near ANY traffic point
    UPDATE vector.road_maharashtra r
    SET traffic_factor = t.max_factor,
        live_cost_s = r.cost_s * t.max_factor
    FROM (
        SELECT r2.gid, MAX(u.factor) as max_factor
        FROM vector.road_maharashtra r2
        JOIN _tmp_traffic_updates u ON r2.geom && ST_Expand(u.geom, u.radius_deg)
        GROUP BY r2.gid
    ) t
    WHERE r.gid = t.gid;
    """
    return sql

# --- Example ---
sample_updates = [
    (19.076, 72.877, 1.8, 1.5),  # Mumbai CST - moderate congestion
    (19.120, 72.850, 3.2, 2.0),  # Andheri - heavy congestion
    (19.000, 72.840, 1.0, 1.5),  # Colaba - free flow
]
print(batch_traffic_update_sql(sample_updates))

### 4.8 — Route Geometry as GeoJSON

**What this does:** Converts route geometries into GeoJSON features with
traffic-colored properties for map visualization.

**Related formula:** Section 3.3 — traffic factor color mapping

In [ ]:
import json

def traffic_factor_to_color(factor: float) -> tuple:
    """
    Map traffic factor to hex color and emoji for visualization.
    
    Color scheme matches Google Maps congestion colors:
    - Green:  Free flow (factor ≤ 1.1)
    - Yellow: Light congestion (1.1 – 1.5)
    - Orange: Moderate congestion (1.5 – 2.0)
    - Red:    Heavy congestion (> 2.0)
    """
    if factor >= 2.0:  return "#DC2626", "🔴 Red   "
    if factor >= 1.5:  return "#F97316", "🟠 Orange"
    if factor >= 1.1:  return "#EAB308", "🟡 Yellow"
    return "#22C55E", "🟢 Green "

# Simulated segment data (in production, these come from PostGIS)
segments = [
    {"vehicle_id": 1, "segment_index": 0, "traffic_factor": 1.0},
    {"vehicle_id": 1, "segment_index": 1, "traffic_factor": 1.3},
    {"vehicle_id": 1, "segment_index": 2, "traffic_factor": 2.1},
    {"vehicle_id": 1, "segment_index": 3, "traffic_factor": 1.7},
]

# Build GeoJSON FeatureCollection
geojson = {"type": "FeatureCollection", "features": []}
print("=== GeoJSON Output ===")
for seg in segments:
    tf = seg["traffic_factor"]
    color, label = traffic_factor_to_color(tf)
    print(f"  Segment {seg['segment_index']}: factor={tf:.2f} → {label} ({color})")
    geojson["features"].append({
        "type": "Feature",
        "properties": {
            "vehicle_id": seg["vehicle_id"],
            "segment_index": seg["segment_index"],
            "traffic_factor": round(tf, 2),
            "traffic_color": color
        },
        "geometry": {"type": "MultiLineString", "coordinates": []}  # From PostGIS
    })

# Show first feature as sample
print(f"\nFull GeoJSON:")
sample_geojson = {"type": "FeatureCollection", "features": geojson["features"][:1]}
print(json.dumps(sample_geojson, indent=2))

=== GeoJSON Output ===
  Segment 0: factor=1.00 → 🟢 Green  (#22C55E)
  Segment 1: factor=1.30 → 🟡 Yellow (#EAB308)
  Segment 2: factor=2.10 → 🔴 Red    (#DC2626)
  Segment 3: factor=1.70 → 🟠 Orange (#F97316)

Full GeoJSON:
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "vehicle_id": 1,
        "segment_index": 0,
        "traffic_factor": 1.0,
        "traffic_color": "#22C55E"
      },
      "geometry": {
        "type": "MultiLineString",
        "coordinates": []
      }
    }
  ]
}


### 4.9 — Geocoding Pipeline (Address → Coordinates)

**What this does:** Demonstrates the geocoding pipeline with cache,
Ola Maps primary, and Nominatim fallback.

**Related section:** Section 3.11

In [ ]:
def validate_coordinates(lat: float, lon: float) -> bool:
    """
    Validate if coordinates are within India's approximate bounds.
    
    India bounds:
    - Latitude:  6.5° to 35.5° N
    - Longitude: 68.0° to 97.5° E
    """
    return (6.5 <= lat <= 35.5) and (68.0 <= lon <= 97.5)


def geocode_pipeline_demo():
    """
    Demonstrates the geocoding pipeline logic.
    
    Real pipeline:
    1. Check JSON cache (geocode_cache.json)
    2. Try Ola Maps API (parallel batch, 5 concurrent)
    3. Fallback to Nominatim/OSM (sequential, 1 req/sec)
    4. Save result to cache
    5. Validate coordinates are within India bounds
    """
    # Simulated cache
    cache = {
        "MG Road, Andheri West, Mumbai": {
            "latitude": 19.137, "longitude": 72.826,
            "source": "krutrim", "confidence": 1.0
        }
    }
    
    test_addresses = [
        "MG Road, Andheri West, Mumbai",  # Should hit cache
        "Bandra Station, Mumbai",          # Cache miss
    ]
    
    print("=== Geocoding Pipeline Demo ===")
    for addr in test_addresses:
        if addr in cache:
            result = cache[addr]
            print(f"  ✓ Cache HIT:  {addr} → ({result['latitude']:.4f}, {result['longitude']:.4f}) [cached]")
        else:
            print(f"  → Cache MISS: {addr} → would call Ola Maps API → Nominatim fallback")
    
    # Validation demo
    print("\n=== Coordinate Validation ===")
    test_coords = [(19.076, 72.877, "Mumbai"), (51.507, -0.127, "London")]
    for lat, lon, name in test_coords:
        valid = validate_coordinates(lat, lon)
        status = "Valid (within India bounds)" if valid else "Invalid (outside India bounds)"
        print(f"  ({lat}, {lon}) → {status}")

geocode_pipeline_demo()

=== Geocoding Pipeline Demo ===
  ✓ Cache HIT:  MG Road, Andheri West, Mumbai → (19.1370, 72.8260) [cached]
  → Cache MISS: Bandra Station, Mumbai → would call Ola Maps API → Nominatim fallback

=== Coordinate Validation ===
  (19.076, 72.877) → Valid (within India bounds)
  (51.507, -0.127) → Invalid (outside India bounds)


### 4.10 — Delivery Status Classification

**What this does:** Classifies delivery timing status based on
arrival time relative to the delivery deadline.

**Related formula:** Section 3.10

In [ ]:
def classify_delivery_status(arrival_min: int, deadline_min: int) -> str:
    """
    Classify delivery timing status.
    
    Rules:
    - IN_BUFFER: Arrival ≤ deadline - 60 min (well within window)
    - ON TIME:   deadline - 60 < arrival ≤ deadline (cutting it close)
    - LATE:      Arrival > deadline
    """
    if arrival_min <= (deadline_min - 60):
        return "IN_BUFFER"
    elif arrival_min <= deadline_min:
        return "ON TIME"
    else:
        return "LATE"


def min_to_clock(minutes: int) -> str:
    """Convert minutes from midnight to HH:MM format."""
    hours = minutes // 60
    mins = minutes % 60
    if hours >= 24:
        return f"{hours - 24:02d}:{mins:02d} (+1 Day)"
    return f"{hours:02d}:{mins:02d}"


# --- Test ---
deadline = 600  # 10:00 AM
test_arrivals = [510, 555, 570, 600, 615]  # 08:30, 09:15, 09:30, 10:00, 10:15

print("=== Delivery Status Classification ===")
for arrival in test_arrivals:
    status = classify_delivery_status(arrival, deadline)
    diff = deadline - arrival
    diff_label = f"{abs(diff)} min {'before' if diff >= 0 else 'after'} deadline"
    print(f"  Deadline={min_to_clock(deadline)} | Arrival={min_to_clock(arrival)} → "
          f"{status:<10s} ({diff_label})")

=== Delivery Status Classification ===
  Deadline=10:00 | Arrival=08:30 → IN_BUFFER  (90 min before deadline)
  Deadline=10:00 | Arrival=09:15 → IN_BUFFER  (45 min before deadline)
  Deadline=10:00 | Arrival=09:30 → ON TIME    (30 min before deadline)
  Deadline=10:00 | Arrival=10:00 → ON TIME    (0 min before deadline)
  Deadline=10:00 | Arrival=10:15 → LATE       (15 min after deadline)


---

## 5. System Specification & Tech Handoff

This section is the **developer reference** — everything needed to build
or maintain the system, independent of any specific codebase.

---

### 5.1 System Inputs

| Input | Format | Required Columns | Defaults |
|-------|--------|-------------------|----------|
| **Delivery Manifest** | CSV/Excel | `id`, `latitude`, `longitude` | — |
| | | `parcel_weight` (kg) | 20 |
| | | `service_time` (min) | 10 |
| | | `window_start` (min from midnight) | 420 (7:00 AM) |
| | | `window_end` (min from midnight) | 600 (10:00 AM) |
| **Warehouse** | lat/lon | — | Mumbai (19.0725, 72.8724) |
| **Fleet** | Array | `name`, `capacity_kg`, `cost_per_km`, `shift_start`, `shift_end` | 10 vehicles (see below) |

---

### 5.2 Telemetry Sources & Formulas

#### Traffic Congestion Factor

| Source | Formula | Auth |
|--------|---------|------|
| Google Routes API | `duration / staticDuration` | OAuth2 Service Account |
| TomTom Flow API | `freeFlowSpeed / currentSpeed` | API Key |

Bounds: `[0.8, 10.0]`. Default: `1.0` (free flow).

#### Weather Severity & Road Penalty

| Severity | Rainfall (mm/hr) | Penalty Factor |
|----------|-----------------|----------------|
| None | < 2.5 | 1.0× |
| Moderate | 2.5 – 7.5 | 3.0× |
| Heavy | ≥ 7.5 | 10.0× |

#### Combined Road Cost
```
live_cost_s = cost_s × max(traffic_factor, weather_penalty_factor)
```

---

### 5.3 Optimization Constraints

| Constraint | Type | Value |
|------------|------|-------|
| Vehicle Capacity | Hard (soft penalty) | `capacity_kg` per vehicle, penalty 1M/kg |
| Time Windows | Soft upper bound | `window_end` per parcel, penalty 100K/min |
| Shift Duration | Soft upper bound | `shift_end` + 60min overtime, penalty 50K/min |
| Drop Penalty | Quasi-hard | 1,000,000,000 per undelivered parcel |
| Warehouse Loading | Fixed | 10 minutes at depot before departure |
| Min Travel Time | Floor | 1 minute between any two different nodes |
| Cross-Midnight | Auto-normalize | If `shift_end < shift_start`, add 1440 |

---

### 5.4 Default Fleet Configuration (10 Vehicles)

| Vehicle | Capacity (kg) | Cost/km (₹) | Shift Start | Shift End |
|---------|--------------|-------------|-------------|----------|
| Vehicle 1 | 175 | 15 | 09:00 | 18:00 |
| Vehicle 2 | 261 | 20 | 09:00 | 18:00 |
| Vehicle 3 | 348 | 25 | 07:00 | 15:00 |
| Vehicle 4 | 156 | 12 | 07:00 | 18:00 |
| Vehicle 5 | 178 | 15 | 09:00 | 17:00 |
| Vehicle 6 | 142 | 12 | 08:00 | 18:00 |
| Vehicle 7 | 118 | 10 | 08:00 | 21:00 |
| Vehicle 8 | 125 | 10 | 07:00 | 20:00 |
| Vehicle 9 | 200 | 12 | 07:00 | 19:00 |
| Vehicle 10 | 180 | 14 | 08:00 | 20:00 |

---

### 5.5 Geocoding Pipeline

```
Input Address → Cache Check → Ola Maps API → Nominatim Fallback → (lat, lon)
```

| Step | Details |
|------|---------|
| Cache | JSON file (`geocode_cache.json`), keyed by raw address string |
| Ola Maps | Parallel batch (5 concurrent), `api.olamaps.io/places/v1/geocode` |
| Nominatim | Sequential (1 req/sec rate limit), `nominatim.openstreetmap.org/search` |
| Validation | India bounds: 6.5°–35.5°N, 68°–97.5°E |

---

### 5.6 Expected System Output

#### Route Results Payload
```json
{
  "vehicles": [
    {
      "vehicle_id": 1,
      "stations": [
        {"station_id": "P001", "arrival_time": "09:42", "status": "ON TIME"}
      ],
      "route_geometry": {"type": "FeatureCollection", "features": [
        {"properties": {"vehicle_id": 1, "segment_index": 0,
                        "traffic_factor": 1.3, "traffic_color": "#EAB308"},
         "geometry": {"type": "MultiLineString", "coordinates": [...]}}
      ]},
      "total_distance": 71.88,
      "total_cost": 1078.2,
      "weight_carried": 174,
      "capacity": 175,
      "utilization": 99.4,
      "work_duration": 143,
      "color": "#FF6B6B",
      "clock_in": "09:00",
      "clock_out": "11:23"
    }
  ],
  "summary": {
    "total_distance": 676.21,
    "total_cost": 9723.30,
    "total_parcels": 53,
    "total_fleets": 8
  },
  "undelivered_parcels": [
    {"station_id": "P054", "reason": "Capacity/time constraints",
     "latitude": 19.05, "longitude": 72.88, "parcel_weight": 30}
  ],
  "weather_alerts": [
    {"station_id": "P012", "lat": 19.12, "lon": 72.85,
     "rain_mm": 9.3, "severity": "heavy",
     "description": "Heavy Rain (9.3 mm/hr)"}
  ],
  "rerouted_vehicles": [3, 7]
}
```

#### Delivery Status Values

| Status | Condition |
|--------|-----------|
| `IN_BUFFER` | Arrival ≤ deadline − 60 min |
| `ON TIME` | deadline − 60 < arrival ≤ deadline |
| `LATE` | Arrival > deadline |

---

### 5.7 API Endpoints

| Method | Endpoint | Purpose |
|--------|----------|---------|
| POST | `/api/upload` | Upload CSV/Excel delivery data |
| POST | `/api/compute` | Trigger route optimization |
| GET | `/api/results` | Retrieve optimized routes |
| POST | `/api/refresh-traffic` | Re-query traffic + regenerate routes |
| POST | `/api/reoptimize` | Re-order stops (fixed assignments) |
| POST | `/api/auto-reoptimize` | Toggle auto re-optimization ON/OFF |
| GET | `/api/download-report` | Download Excel report |

---

### 5.8 Database Schema (Required Tables)

| Table | Purpose |
|-------|---------|
| `vector.road_maharashtra` | Road network (~843K segments) with traffic columns |
| `vector.main_road_nodes` | Pre-computed connected component nodes with GiST index |
| `vector.station_node_map` | Uploaded delivery stations snapped to road nodes |
| `vector.distance_matrix` | Pre-computed N×N shortest path costs |
| `vector.route_geometries` | Per-segment route geometries with traffic factors |
| `vector.fleet_vehicles` | Configurable fleet (capacity, cost, shifts) |
| `vector.unassigned_parcels` | Parcels that couldn't be assigned |

---

### 5.9 Performance Benchmarks

| Component | Target | Actual |
|-----------|--------|--------|
| Station Snapping (50 stations) | < 2s | ~1s |
| Distance Matrix (50 nodes) | < 20s | ~15s |
| VRP Solver (50 parcels, 10 vehicles) | < 15s | ~10s |
| Route Geometry (8 routes) | < 5s | ~3s |
| Traffic + Weather Sync | < 10s | ~5s |
| **Total Pipeline** | **< 60s** | **~39s** |